<a href="https://colab.research.google.com/github/PatrickCalorioCarvalho/CA015ICMineracaoDeDados202601/blob/main/Aula_FeautureSelection_Alzheimer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Base de dados: Alzheimer's Disease Dataset

Link: https://www.kaggle.com/datasets/rabieelkharoua/alzheimers-disease-dataset


In [1]:
# Bibliotecas
import pandas as pd

# sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report
from sklearn.feature_selection import SelectKBest, mutual_info_classif

In [2]:
# Carregamento da base de dados
df = pd.read_csv('alzheimers_disease_data.csv')

In [3]:
# Tipos de dados
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2149 entries, 0 to 2148
Data columns (total 35 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   PatientID                  2149 non-null   int64  
 1   Age                        2149 non-null   int64  
 2   Gender                     2149 non-null   int64  
 3   Ethnicity                  2149 non-null   int64  
 4   EducationLevel             2149 non-null   int64  
 5   BMI                        2149 non-null   float64
 6   Smoking                    2149 non-null   int64  
 7   AlcoholConsumption         2149 non-null   float64
 8   PhysicalActivity           2149 non-null   float64
 9   DietQuality                2149 non-null   float64
 10  SleepQuality               2149 non-null   float64
 11  FamilyHistoryAlzheimers    2149 non-null   int64  
 12  CardiovascularDisease      2149 non-null   int64  
 13  Diabetes                   2149 non-null   int64

### Pré-processamento
Após umam breve análise:
  * não pé necessário aplicar substituições (NaN);
  * Não é necessário tranformações (OHE)

In [4]:
# Preparação da base
## Remoção das colunas irrelevantes: [PatientID, DoctorInCharge]

cols_remover = ['PatientID', 'DoctorInCharge']
df = df.drop(columns=cols_remover, axis=1)

In [5]:
df['Diagnosis'].value_counts()

,count
Diagnosis,
0,1389
1,760


In [6]:
# Separação de Treino e Teste
X = df.drop(columns = ['Diagnosis'], axis=1)
y = df['Diagnosis']

# Aplica a separação com train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size = 0.30,
    stratify = y
)

In [7]:
X_train.shape, X_test.shape

((1504, 32), (645, 32))

In [8]:
# Aplicar a normalização no treino e teste
## Como todas as colunas são numéricas, n
## Não é necessário filtrar com select_dtypes
numeric_cols = X_train.columns.tolist()

# Normalização
scaler = StandardScaler()
X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])

In [9]:
# Aplica somente o transform no test
X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])

In [10]:
# Avalia o modelo com todas as features
clf = DecisionTreeClassifier(random_state= 42)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.94      0.92      0.93       417
           1       0.85      0.89      0.87       228

    accuracy                           0.91       645
   macro avg       0.90      0.90      0.90       645
weighted avg       0.91      0.91      0.91       645



In [11]:
# Aplicar a seleção de características filter
## Mutual Information

# Especificar o conjunto minimo de features
tam_f = 10

selector = SelectKBest(
    score_func = mutual_info_classif,
    k = tam_f
)

# Treinar a seleção
selector.fit(X_train, y_train)

SelectKBest(score_func=<function mutual_info_classif at 0x7e6892f94900>)

In [12]:
# Mostrar as features
selected_features = X_train.columns[selector.get_support()].tolist()

In [13]:
selected_features

['EducationLevel',
 'Smoking',
 'PhysicalActivity',
 'SleepQuality',
 'SystolicBP',
 'MMSE',
 'FunctionalAssessment',
 'MemoryComplaints',
 'BehavioralProblems',
 'ADL']

In [14]:
X_train[selected_features]

,EducationLevel,Smoking,PhysicalActivity,SleepQuality,SystolicBP,MMSE,FunctionalAssessment,MemoryComplaints,BehavioralProblems,ADL
701,0.797101,-0.631720,0.513020,0.473695,-0.431437,0.342554,-0.337913,-0.518843,-0.433582,-0.939808
1624,0.797101,-0.631720,1.367626,0.701187,1.688588,-1.238518,-0.517889,-0.518843,-0.433582,-1.125031
1839,-1.400589,-0.631720,1.457816,-0.775770,-1.317993,0.558121,1.668565,-0.518843,-0.433582,0.913514
1215,0.797101,-0.631720,-0.496062,0.856996,-0.123070,-1.524398,0.832802,-0.518843,-0.433582,-0.751149
793,0.797101,-0.631720,-0.798896,-1.549499,1.688588,-0.707036,-0.426754,-0.518843,-0.433582,0.303853
...,...,...,...,...,...,...,...,...,...,...
727,-0.301744,-0.631720,1.007930,1.352720,0.609302,1.713736,-0.456390,-0.518843,2.306367,-0.560215
1190,-1.400589,-0.631720,-0.516692,-0.830449,1.572950,-0.966721,-0.332221,1.927364,-0.433582,1.303655
1808,1.895946,-0.631720,-0.990503,0.512028,-0.161616,-0.912593,1.688070,-0.518843,2.306367,0.955715
2059,-0.301744,-0.631720,0.678033,1.335269,-0.778351,-1.254583,-0.453196,-0.518843,-0.433582,-0.040461


In [15]:
# Re-treinar o modelo com as selected_features
clf_filter = DecisionTreeClassifier(random_state = 42)
clf_filter.fit(X_train[selected_features], y_train)
y_pred_filter = clf_filter.predict(X_test[selected_features])

In [16]:
# Avalia o modelo com as features selecionadas
print(classification_report(y_test, y_pred_filter ))

              precision    recall  f1-score   support

           0       0.95      0.93      0.94       417
           1       0.87      0.92      0.90       228

    accuracy                           0.92       645
   macro avg       0.91      0.92      0.92       645
weighted avg       0.93      0.92      0.92       645



In [17]:
# Avalia o modelo com as features selecionadas
print(classification_report(y_test, y_pred ))

              precision    recall  f1-score   support

           0       0.94      0.92      0.93       417
           1       0.85      0.89      0.87       228

    accuracy                           0.91       645
   macro avg       0.90      0.90      0.90       645
weighted avg       0.91      0.91      0.91       645



* Qual o subconjunto mínimo para k (SelectKBest)?
* Mutual Information é a melhor abordagem?